# 06. Власний пайплайн

Дві частини, оцінені окремо, щоб розділити їхній внесок:

- **A. Дані / інференс** — без перенавчання, на найкращій моделі з `04_yolo`: більша роздільність на інференсі, TTA, нарізка кадру на тайли (як у SAHI), часове перерахування оцінок за сусідніми кадрами (як у Seq-NMS).
- **B. Модель** — більша модель YOLO11m з найкращими налаштуваннями з `04_yolo`, далі ті самі стратегії інференсу.

Фінальний варіант обирається за `mAP_test_like` (кліпи val шириною ≤ 1920, як у test).

In [ ]:
SMOKE_TEST = False

In [ ]:
MODEL_B = "yolo11m.pt"
EPOCHS_B = 30
TILE_GRID = 2
TILE_OVERLAP = 0.2
TEMPORAL_IOU = 0.5
SEED = 42
WORKERS = 16
SELECT_BY = "mAP_test_like"

## 1. Шляхи, дані, модель з `04_yolo`

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from ultralytics import YOLO

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    for name in ("hw2_eval.py", "hw2_ultra.py"):
        sys.path.append(str(next(Path("/kaggle/input").glob(f"**/{name}")).parent))
import hw2_eval as E
import hw2_ultra as U

DATA, PREPARED, RUNS = U.paths()
OUT = RUNS / ("smoke_pipeline" if SMOKE_TEST else "pipeline")
OUT.mkdir(parents=True, exist_ok=True)

yolo_runs = ["baseline", "exp1_imgsz1280", "exp2_nwd"]
yolo_dir = RUNS / "yolo"
if SMOKE_TEST and not (yolo_dir / "baseline" / "metrics.json").exists():
    yolo_dir, yolo_runs = RUNS / "yolo", ["smoke_baseline"]
BASE_RUN = U.best_run(yolo_dir, yolo_runs, SELECT_BY)
BASE = json.loads((yolo_dir / BASE_RUN / "metrics.json").read_text())["config"]
print(U.gpu_name(), BASE_RUN, BASE, sep="\n")

In [ ]:
val = U.val_frames(DATA)
test = U.test_frames(DATA)
val_sub = val[val["pos"] % U.VAL_STEP == 0]
if SMOKE_TEST:
    EPOCHS_B = 1
    val, test, val_sub = val.head(6), test.head(4), val_sub.head(8)
gt = E.load_gt(DATA / "val" / "annotations_coco.json")

## 2. Стратегії інференсу

- `base` — як у `04_yolo`;
- `imgsz+` — роздільність інференсу ×1,33 (дрібні об'єкти стають більшими для моделі);
- `tta` — вбудований TTA Ultralytics (віддзеркалення + масштаби);
- `tiles` — повний кадр + сітка 2×2 тайлів з перекриттям 20%, кожен тайл у моделі на тій самій роздільності, тобто з ~1,8× збільшенням; рамки зводимо в координати кадру й об'єднуємо NMS з IoU 0,6 по класах;
- `+temporal` — до найкращого з попередніх: оцінка кожної рамки усереднюється з оцінками рамок того самого класу з IoU ≥ 0,5 на попередньому й наступному кадрі (відсутня пара = 0). Стабільні в часі рамки зберігають оцінку, поодинокі — втрачають.

In [ ]:
def bigger(imgsz):
    return int(round(imgsz * 4 / 3 / 32) * 32)


def run_strategies(model, imgsz, tag):
    variants = {
        "base": lambda f: U.predict(model, f, imgsz),
        "imgsz+": lambda f: U.predict(model, f, bigger(imgsz)),
        "tta": lambda f: U.predict(model, f, imgsz, augment=True),
        "tiles": lambda f: U.predict_tiled(model, f, imgsz, TILE_GRID, TILE_OVERLAP),
    }
    rows, predictions = {}, {}
    for name, fn in variants.items():
        elapsed = U.timer()
        predictions[name] = fn(val)
        rows[name] = {**E.report(gt, predictions[name])[0].to_dict(), "val_min": elapsed()}
        print(tag, name, round(rows[name][SELECT_BY], 4))
    best = max(rows, key=lambda n: rows[n][SELECT_BY])
    name = f"{best}+temporal"
    predictions[name] = U.temporal_rescore(predictions[best], val, TEMPORAL_IOU)
    rows[name] = E.report(gt, predictions[name])[0].to_dict()
    table = pd.DataFrame(rows).T
    table.to_csv(OUT / f"strategies_{tag}.csv")
    return table, {**variants, name: lambda f: U.temporal_rescore(variants[best](f), f, TEMPORAL_IOU)}

## 3. Частина A: інференс на моделі з `04_yolo`

In [ ]:
model_a = YOLO(yolo_dir / BASE_RUN / "weights" / "best.pt")
table_a, variants_a = run_strategies(model_a, BASE["imgsz"], "A")
table_a[["mAP", "AP50", "AP_small", "mAP_test_like", "val_min"]].round(4)

## 4. Частина B: більша модель

YOLO11m з роздільністю й втратою найкращого запуску `04_yolo`, той самий рецепт, 30 епох. Batch зменшено вдвічі через 8 ГБ пам'яті GPU (Ultralytics накопичує градієнти до номінального batch 64, тож крок оптимізатора той самий).

In [ ]:
import torch
import ultralytics.utils.loss as yolo_loss
from ultralytics.utils.metrics import bbox_iou

NWD = BASE.get("nwd", 0.0)
NWD_C = BASE.get("nwd_c", 3.0)


def iou_with_nwd(box1, box2, xywh=False, **kwargs):
    iou = bbox_iou(box1, box2, xywh=xywh, **kwargs)
    center = ((box1[:, :2] + box1[:, 2:]) - (box2[:, :2] + box2[:, 2:])) / 2
    size = (box1[:, 2:] - box1[:, :2]) - (box2[:, 2:] - box2[:, :2])
    w2 = (center ** 2).sum(1, keepdim=True) + (size ** 2).sum(1, keepdim=True) / 4
    return (1 - NWD) * iou + NWD * torch.exp(-w2.clamp(min=1e-7).sqrt() / NWD_C)


yolo_loss.bbox_iou = iou_with_nwd if NWD > 0 else bbox_iou

out_b = OUT / "yolo11m"
data_yaml = U.write_data_yaml(PREPARED, list(val_sub["path"]), out_b)
elapsed = U.timer()
if U.resumable(out_b):
    YOLO(out_b / "weights" / "last.pt").train(resume=True)
elif not (out_b / "weights" / "last.pt").exists():
    YOLO(MODEL_B).train(data=str(data_yaml), imgsz=BASE["imgsz"], batch=max(2, BASE["batch"] // 2), epochs=EPOCHS_B,
                        seed=SEED, deterministic=True, workers=WORKERS, cache="disk", project=str(OUT), name="yolo11m",
                        exist_ok=True, fraction=0.002 if SMOKE_TEST else 1.0, plots=True, verbose=False)
train_minutes_b = elapsed()
U.history(out_b)[["epoch", "mAP", "AP50"]].round(4).tail(5)

In [ ]:
model_b = YOLO(out_b / "weights" / "best.pt")
table_b, variants_b = run_strategies(model_b, BASE["imgsz"], "B")
table_b[["mAP", "AP50", "AP_small", "mAP_test_like", "val_min"]].round(4)

## 5. Внесок даних / інференсу і моделі окремо

In [ ]:
cols = ["mAP", "AP50", "AP_small", "AP_medium", "mAP_test_like"]
comparison = pd.concat({f"YOLO11s ({BASE_RUN})": table_a[cols], "YOLO11m": table_b[cols]}, axis=1)
comparison.to_csv(OUT / "comparison.csv")
comparison.round(4)

## 6. Фінальний варіант і сабміт

In [ ]:
candidates = {("A", n): table_a.loc[n, SELECT_BY] for n in table_a.index}
candidates |= {("B", n): table_b.loc[n, SELECT_BY] for n in table_b.index}
part, variant = max(candidates, key=candidates.get)
final_model = model_a if part == "A" else model_b
final_variants = variants_a if part == "A" else variants_b
final_imgsz = BASE["imgsz"]

test_detections = final_variants[variant](test)
final = OUT / "final"
final.mkdir(exist_ok=True)
E.write_submission(test_detections, DATA / "test_images.csv", final / "submission.csv")
speed = U.ms_per_image(final_model, list(val["path"]), final_imgsz)
choice = {"part": part, "variant": variant, "base_run": BASE_RUN, "model": "yolo11s" if part == "A" else "yolo11m",
          "imgsz": final_imgsz, "nwd": NWD, "train_min_b": train_minutes_b, "ms_per_image_base": speed,
          "val": (table_a if part == "A" else table_b).loc[variant].to_dict()}
(final / "metrics.json").write_text(json.dumps(choice, indent=1))
(final / "test_predictions.json").write_text(json.dumps(test_detections))
pd.Series(choice["val"]).round(4), choice["model"], variant

## Джерела

**Бібліотеки**
- Ultralytics 8.3.40 — YOLO11s/11m, TTA (`augment=True`): https://docs.ultralytics.com/modes/predict/
- torchvision — `batched_nms`, `box_iou`; OpenCV — читання кадрів для тайлів.

**Методи зі статей**
- Нарізка на тайли з перекриттям і злиття з повним кадром — SAHI, Akyon et al., arXiv:2202.06934 (у нас простіша версія: фіксована сітка 2×2 і NMS).
- Часова узгодженість детекцій — Seq-NMS, Han et al., arXiv:1602.08465; T-CNN, arXiv:1604.02532 (у нас простіша версія: усереднення з парами лише на сусідніх кадрах).
- NWD — arXiv:2110.13389 (якщо найкращий запуск `04_yolo` — `exp2_nwd`).

**Наші рішення:** набір стратегій, сітка 2×2 з перекриттям 20%, поріг злиття 0,6, правило усереднення оцінок (відсутній сусід = 0), YOLO11m на 30 епох, вибір за `mAP_test_like`.